<a href="https://colab.research.google.com/github/Shruti022/Finance_chatbot/blob/main/Finance_chatbot_aml_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q google-generativeai sentence-transformers faiss-cpu yfinance pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 82.9 MB/s eta 0:00:00


In [3]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.2 MB/s eta 0:00:00


In [4]:
from groq import Groq

from google.colab import userdata
import google.generativeai as genai

# Fetch the secret by name
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

client = Groq(api_key=GROQ_API_KEY)

def generate_response(prompt, max_tokens=400):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",  # Best quality
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content
    except Exception as e:
        # Fallback to faster model if rate limited
        if "429" in str(e):
            print("⚠️ Rate limited, switching to faster model...")
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",  # Unlimited backup
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content
        raise e

print("✓ Groq ready with auto-fallback!")


✓ Groq ready with auto-fallback!


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import pandas as pd
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import yfinance as yf
import re

# Upload your files first, then load
df_rag = pd.read_pickle("/content/drive/MyDrive/Sem 1/AML/project/financeqa_df.pkl")
index_rag = faiss.read_index("/content/drive/MyDrive/Sem 1/AML/project/financeqa_index.faiss")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"✓ RAG: {len(df_rag)} docs, {index_rag.ntotal} vectors")

# Yahoo Finance helpers
def get_ticker_data(ticker):
    try:
        t = yf.Ticker(ticker)
        info = t.info
        return {
            "symbol": ticker,
            "price": info.get("currentPrice") or info.get("regularMarketPrice"),
            "pe": info.get("trailingPE"),
            "forward_pe": info.get("forwardPE"),
            "sector": info.get("sector"),
            "market_cap": info.get("marketCap"),
            "52w_high": info.get("fiftyTwoWeekHigh"),
            "52w_low": info.get("fiftyTwoWeekLow"),
        }
    except:
        return None

def extract_ticker(text):
    candidates = re.findall(r"\b[A-Z]{2,5}\b", text)
    blacklist = {"WHAT", "IS", "ARE", "THE", "AND", "ETF", "STOCK", "HOW", "WHY", "PE", "ROE", "EPS", "TELL", "ABOUT"}
    tickers = [c for c in candidates if c not in blacklist]
    return tickers[0] if tickers else None

def retrieve_context(query, k=3):
    q_emb = embed_model.encode([query])
    q_emb = np.array(q_emb, dtype="float32")
    faiss.normalize_L2(q_emb)
    scores, idxs = index_rag.search(q_emb, k)
    snippets = [df_rag.iloc[i]["CONTEXT"] for i in idxs[0]]
    return "\n".join(snippets)

print("✓ Yahoo Finance + RAG ready")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ RAG: 3705 docs, 3705 vectors
✓ Yahoo Finance + RAG ready


In [17]:
class FundamentalAnalyst:
    def __init__(self):
        self.name = "Fundamental Analyst"
        self.role = (
            "You are a fundamental analyst at a top investment firm. "
            "You analyze companies using financial ratios from live data and reports. "
            "Focus on: PE ratio, profit margins, debt levels, ROE. "
            "Be specific with numbers."
        )

    def analyze(self, question, ticker=None):
        # Get both RAG and live data
        context_parts = []

        if ticker:
            data = get_ticker_data(ticker)
            if data:
                context_parts.append(f"""
LIVE FUNDAMENTAL DATA for {ticker}:
- PE Ratio: {data['pe']}
- Forward PE: {data['forward_pe']}
- Market Cap: ${data['market_cap']:,} if data['market_cap'] else 'N/A'
- Sector: {data['sector']}
""")

        # Add RAG as supplementary context
        rag_context = retrieve_context(question + " fundamentals ratios", k=2)
        if rag_context:
            context_parts.append(f"INDUSTRY CONTEXT:\n{rag_context}")

        full_context = "\n\n".join(context_parts)

        prompt = f"""{self.role}

QUESTION: {question}

{full_context}

Provide your analysis focusing on:
1. Key financial ratios (use LIVE DATA numbers if available)
2. Financial health assessment
3. Your bullish or bearish stance

Keep it under 150 words. Be direct and opinionated."""

        return generate_response(prompt)

agent1 = FundamentalAnalyst()


print(agent1.analyze("Should I invest in AAPL?"))


I'm analyzing AARTIIND, not AAPL. Based on the provided data, here's my assessment:

AARTIIND's debt level is moderate, with long-term borrowings at 634.71. The company's ROE is approximately 12.8% (calculated using Total Reserves and Surplus). Without live data on profit margins and PE ratio, I'll focus on the available information. 

Given the moderate debt and decent ROE, I'm neutral on AARTIIND. However, without current market data, it's challenging to make a definitive call. If I had to choose, I'd be slightly bullish due to the company's manageable debt and respectable return on equity.


In [18]:
class MarketDataAnalyst:
    def __init__(self):
        self.name = "Market Data Analyst"
        self.role = (
            "You are a market data analyst specializing in real-time price action and trends. "
            "You analyze: current price, 52-week highs/lows, sector performance, PE vs sector average. "
            "You care about momentum, valuation relative to peers, and market sentiment."
        )

    def analyze(self, question, ticker):
        if not ticker:
            return "No ticker symbol detected."

        data = get_ticker_data(ticker)
        if not data:
            return f"Could not retrieve data for {ticker}."

        market_summary = f"""
Symbol: {data['symbol']}
Current Price: ${data['price']}
PE Ratio: {data['pe']}
Sector: {data['sector']}
52-Week High: ${data['52w_high']}
52-Week Low: ${data['52w_low']}
"""

        prompt = f"""{self.role}

QUESTION: {question}

LIVE MARKET DATA:
{market_summary}

Provide your analysis focusing on:
1. Price positioning (near highs/lows?)
2. Valuation assessment (PE ratio context)
3. Market momentum and sector trends

Keep it under 150 words."""

        return generate_response(prompt)

agent2 = MarketDataAnalyst()
print(agent2.analyze("Should I invest in AAPL?", "AAPL"))


Based on live market data, AAPL's current price of $274.61 is near its 52-week high of $288.62, indicating a relatively strong price positioning. The PE ratio of 36.76 is high, suggesting AAPL may be overvalued compared to its peers. In the technology sector, a high PE ratio may be common, but it's essential to consider the sector average for context. Without the sector average PE, it's difficult to make a definitive valuation assessment. Market momentum appears positive, with AAPL trading near its highs. However, caution is advised due to the high valuation and potential for mean reversion. Further analysis of sector trends and peer comparison is necessary to make an informed investment decision.


In [19]:
class RiskAnalyst:
    def __init__(self):
        self.name = "Risk Analyst"
        self.role = (
            "You are a risk analyst whose job is to identify what could go wrong. "
            "You focus on: market volatility, regulatory risks, competition, macroeconomic threats. "
            "You play devil's advocate."
        )

    def analyze(self, question, ticker=None):
        ticker_context = f"for {ticker}" if ticker else ""

        prompt = f"""{self.role}

QUESTION: {question}

Provide a risk assessment {ticker_context} covering:
1. Top 3 specific risks
2. Bear case scenario
3. Risk rating: Low/Medium/High

Keep it under 150 words. Be skeptical."""

        return generate_response(prompt)

agent3 = RiskAnalyst()
print(agent3.analyze("Should I invest in AAPL?", "AAPL"))


I'd advise caution with AAPL. Here are the top risks:

1. Regulatory risks: Antitrust lawsuits and potential app store commission changes.
2. Competition: Intensifying rivalry from Chinese tech giants like Huawei and Xiaomi.
3. Market volatility: AAPL's high market cap makes it vulnerable to broad market downturns.

Bear case scenario: A global recession hits, Apple's premium products see declining sales, and regulatory pressures force significant app store commission cuts.

Risk rating: High. As a risk analyst, I'm skeptical about AAPL's ability to maintain its dominance. The potential downsides outweigh the upside, making it a risky investment.


In [20]:
class ChiefAnalyst:
    def __init__(self):
        self.name = "Chief Investment Officer"
        self.role = (
            "You are the Chief Investment Officer synthesizing input from your analyst team. "
            "You provide: BUY/HOLD/SELL rating, conviction score (1-10), and clear reasoning."
        )

    def synthesize(self, question, fund, market, risk):
        prompt = f"""{self.role}

QUESTION: {question}

ANALYST REPORTS:

Fundamental Analyst: {fund}

Market Data Analyst: {market}

Risk Analyst: {risk}

Provide FINAL RECOMMENDATION:
1. Rating: BUY/HOLD/SELL
2. Conviction: X/10
3. Key reasoning (2-3 sentences)

Keep it under 200 words."""

        return generate_response(prompt, max_tokens=500)

moderator = ChiefAnalyst()


In [21]:
def run_analyst_team(question):
    print("="*60)
    print(f"INVESTMENT QUESTION: {question}")
    print("="*60)

    # Extract ticker
    ticker = extract_ticker(question)
    if ticker:
        print(f"📊 Detected ticker: {ticker}\n")

    # Agent 1: Fundamentals
    print(f"💼 {agent1.name} is analyzing...")
    fund_analysis = agent1.analyze(question, ticker)
    print(f"\n{fund_analysis}\n")
    print("-"*60)

    # Agent 2: Market Data
    print(f"📈 {agent2.name} is analyzing...")
    market_analysis = agent2.analyze(question, ticker)
    print(f"\n{market_analysis}\n")
    print("-"*60)

    # Agent 3: Risk
    print(f"⚠️  {agent3.name} is analyzing...")
    risk_analysis = agent3.analyze(question, ticker)
    print(f"\n{risk_analysis}\n")
    print("-"*60)

    # Moderator synthesizes
    print(f"🎯 {moderator.name} is synthesizing...")
    final_rec = moderator.synthesize(question, fund_analysis, market_analysis, risk_analysis)
    print(f"\n{final_rec}\n")
    print("="*60)

    return {
        "fundamental": fund_analysis,
        "market": market_analysis,
        "risk": risk_analysis,
        "final": final_rec
    }

# TEST IT
result = run_analyst_team("Should I invest in TSLA?")


INVESTMENT QUESTION: Should I invest in TSLA?
📊 Detected ticker: TSLA

💼 Fundamental Analyst is analyzing...

I'm bearish on TSLA. The PE ratio of 333.25 is excessively high, indicating overvaluation. The net profit margin of 1.92% is relatively low. The debt-to-equity ratio of 0.2 is manageable, but the return on equity (ROE) of 1.64% is underwhelming. With a market cap of over $1.6 trillion, I expect more robust profitability. The forward PE of 217.91 still suggests significant overvaluation. Overall, TSLA's financial health is questionable, and its valuation is stretched. I would avoid investing in TSLA until its financials improve and its valuation becomes more reasonable.

------------------------------------------------------------
📈 Market Data Analyst is analyzing...

Based on live market data, TSLA is currently trading near its 52-week high of $491.5, indicating a strong price positioning. However, the PE ratio of 333.2517 is significantly elevated, suggesting overvaluation re

In [22]:
test_questions = [
    "Should I buy AAPL?",
    "Is NVDA overvalued?",
    "Tell me about MSFT stock",
]

for q in test_questions:
    run_analyst_team(q)
    print("\n\n")


INVESTMENT QUESTION: Should I buy AAPL?
📊 Detected ticker: AAPL

💼 Fundamental Analyst is analyzing...

I'm bearish on AAPL. The PE ratio of 36.76 is high, indicating overvaluation. The forward PE of 30.13 suggests some correction, but still above average. Without profit margin and debt level data, I'm cautious. The market cap of $4,075 billion is massive, making it harder to grow. I'd like to see lower PE and more financial data before investing. AAPL's valuation seems stretched, making it a sell for me.

------------------------------------------------------------
📈 Market Data Analyst is analyzing...

Based on the live market data, AAPL is currently trading near its 52-week high, about 4.8% below the peak. The PE ratio of 36.76 is relatively high, indicating potential overvaluation compared to its sector average. In the technology sector, AAPL's valuation premium may be a concern. Considering market momentum, AAPL's price is strong, but the proximity to 52-week highs and high PE rat

In [23]:
1

1

In [16]:
1

1